# Fase 4 — Modelagem

Produto de referência: P0131. Treino até 03/10/2019, teste de 04/10 a 31/10/2019 (H = 28).

## Infraestrutura de avaliação

In [1]:
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.dados import carregar_demanda
from src.metricas import vies, wmape
from src.preparacao import H, remover_descontinuados, separar_treino_teste, serie_produto

SEED = 42
np.random.seed(SEED)
RAIZ = Path.cwd().parent
CAMINHO_FIGURAS = RAIZ / "relatorios" / "figuras"
PRODUTO = "P0131"

painel = remover_descontinuados(carregar_demanda())
serie_p0131 = serie_produto(painel, PRODUTO)
treino, teste = separar_treino_teste(serie_p0131)
real_teste = teste["unidades"]
print(f"Treino: {treino.index.min().date()} a {treino.index.max().date()} ({len(treino)} dias)")
print(f"Teste:  {teste.index.min().date()} a {teste.index.max().date()} ({len(teste)} dias)")

Treino: 2017-01-02 a 2019-10-03 (1005 dias)
Teste:  2019-10-04 a 2019-10-31 (28 dias)


In [2]:
placar = {}
previsoes = {}


def avaliar(nome, previsao):
    previsao = pd.Series(np.clip(np.asarray(previsao, dtype=float), 0, None), index=real_teste.index)
    placar[nome] = {"wmape": wmape(real_teste, previsao), "vies": vies(real_teste, previsao)}
    previsoes[nome] = previsao
    print(f"{nome:<32} | WMAPE: {placar[nome]['wmape']:6.2f}% | Viés: {placar[nome]['vies']:+6.2f}%")

Checagem das funções com valores de resultado conhecido (não entram no placar).

In [3]:
real_ex = np.array([100, 200, 100])
print(f"Previsão perfeita      -> WMAPE {wmape(real_ex, real_ex):.2f}% | viés {vies(real_ex, real_ex):+.2f}%")
print(f"Tudo 10% acima         -> WMAPE {wmape(real_ex, real_ex * 1.1):.2f}% | viés {vies(real_ex, real_ex * 1.1):+.2f}%")
print(f"Tudo 10% abaixo        -> WMAPE {wmape(real_ex, real_ex * 0.9):.2f}% | viés {vies(real_ex, real_ex * 0.9):+.2f}%")
print(f"+50 e -50 se compensam -> WMAPE {wmape(real_ex, [150, 150, 100]):.2f}% | viés {vies(real_ex, [150, 150, 100]):+.2f}%")
print(f"Clip: previsão [-10, 5, 20] vira {np.clip([-10, 5, 20], 0, None).tolist()}")

Previsão perfeita      -> WMAPE 0.00% | viés +0.00%
Tudo 10% acima         -> WMAPE 10.00% | viés +10.00%
Tudo 10% abaixo        -> WMAPE 10.00% | viés -10.00%
+50 e -50 se compensam -> WMAPE 25.00% | viés +0.00%
Clip: previsão [-10, 5, 20] vira [0, 5, 20]


In [4]:
print(f"Dias avaliados no teste:        {len(real_teste)}")
print(f"Demanda real total no teste:    {real_teste.sum():,.0f} unidades")
print(f"Média diária no teste:          {real_teste.mean():.2f} unidades")

Dias avaliados no teste:        28
Demanda real total no teste:    8,158 unidades
Média diária no teste:          291.36 unidades
